# InsiderFinance + Pineify — Strike Profile với Volume thật

> **Thu thập dữ liệu tự động (live/continuous) giờ nằm ở `scripts/insiderfinance_volume_snapshot.py`**, chạy độc lập ngoài Jupyter (cron/CI/daemon), ghi kết quả vào `data/options/volume/{ticker}_volume_latest.json`. Notebook này giờ chỉ giữ lại các hàm tính toán/vẽ biểu đồ (`normalize_and_calculate_gex`, `build_strike_profile`, plot, export) để tham khảo/khám phá thủ công — các cell fetch/merge InsiderFinance+Pineify đã bị xóa. Muốn chạy lại từ đầu, cần tự cung cấp `raw_options`/`spot` (ví dụ copy logic fetch từ script trên) trước khi chạy các cell bên dưới.

Notebook này đọc dữ liệu công khai được nhúng trong `__NEXT_DATA__` của trang Gamma Exposure (InsiderFinance), chuẩn hóa toàn bộ option contracts, tính GEX theo công thức đang dùng trên trang, tổng hợp theo strike — và bổ sung **Volume thật theo từng contract** lấy từ Pineify (InsiderFinance chỉ có Open Interest, không có Volume).

Kết quả gồm:
- `raw_options`: toàn bộ contracts nhận được từ trang InsiderFinance.
- `options`: contracts đã chuẩn hóa, có GEX và Volume (join từ Pineify).
- `strike_profile`: toàn bộ dữ liệu Strike Profile theo strike, gồm cả `call_volume`/`put_volume`/`total_volume`.
- Các file CSV/JSON trong thư mục `exports/`.

> Chỉ dùng endpoint trang công khai. Cấu trúc Next.js có thể thay đổi; hãy tuân thủ điều khoản sử dụng và quyền phân phối dữ liệu của nhà cung cấp.
>
> Phần Volume gọi tới `https://pineify.app/options-chain` bằng Playwright và chỉ đọc lại response mà trình duyệt tự nhận được (request cần một `x-site-token` ngắn hạn do trang tự lấy khi tải — không phải đăng nhập/paywall). Notebook không cố vượt qua đăng nhập, paywall hay CAPTCHA.

In [1]:
from __future__ import annotations

import html as html_lib
import json
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from IPython.display import HTML, display

TICKER = "QQQ"
MIN_OPEN_INTEREST = 0
PLOT_MODE = "near"  # near: 100 strikes gần spot | all: toàn bộ strikes
PLOT_NEAREST_STRIKES = 100
STRIKE_WINDOW = 100.0  # chỉ giữ các strike trong khoảng spot ± 100 USD
CONTRACT_MULTIPLIER = 100
MOVE_SIZE = 0.01  # GEX cho biến động 1% của underlying
MM_POSITION_MODE = "site_default"  # site_default | always_short | always_long
EXPORT_DIR = Path("exports")
REQUEST_TIMEOUT = 45

# Pineify (nguồn Volume thật theo từng contract — InsiderFinance chỉ có Open Interest).
PINEIFY_URL = "https://pineify.app/options-chain"
SNAPSHOT_URL_TOKEN = "massive/options/snapshot"
PINEIFY_REQUEST_TIMEOUT_MS = 45_000
PINEIFY_HEADLESS = True
PINEIFY_EXPORT_DIR = EXPORT_DIR / "pineify"

pd.set_option("display.max_rows", 1000)
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

## 2. Chuẩn hóa option contracts và tính GEX

Công thức Strike Profile của trang cho mức dịch chuyển 1%:

`GEX = gamma × open_interest × 100 × spot² × 0.01`

Ở chế độ mặc định, Call mang dấu dương và Put mang dấu âm khi không có giá giao dịch để suy luận vị thế market maker.

In [3]:
def _trade_direction(row: pd.Series) -> str:
    # Logic tương ứng trang: thiếu last/price => mid; mid được xem là MM short.
    explicit = row.get("tradeDirection")
    if explicit in {"ask", "above-ask", "bid", "below-bid", "mid"}:
        return explicit

    price = pd.to_numeric(row.get("price", row.get("last", np.nan)), errors="coerce")
    bid = pd.to_numeric(row.get("bid", np.nan), errors="coerce")
    ask = pd.to_numeric(row.get("ask", np.nan), errors="coerce")
    if not np.isfinite(price) or not np.isfinite(bid) or not np.isfinite(ask) or bid <= 0 or ask <= 0 or bid >= ask:
        return "mid"

    midpoint = (bid + ask) / 2
    quarter_spread = 0.25 * (ask - bid)
    if price >= ask:
        return "above-ask"
    if price > midpoint + quarter_spread:
        return "ask"
    if price <= bid:
        return "below-bid"
    if price < midpoint - quarter_spread:
        return "bid"
    return "mid"


def normalize_and_calculate_gex(
    raw: pd.DataFrame,
    ticker: str,
    spot_price: float,
    min_open_interest: int = 0,
    mm_position_mode: str = "site_default",
) -> pd.DataFrame:
    if mm_position_mode not in {"site_default", "always_short", "always_long"}:
        raise ValueError("MM_POSITION_MODE không hợp lệ")

    df = raw.copy()
    numeric_columns = [
        "strike", "expireYear", "expireMonth", "expireDay",
        "gamma", "delta", "openInterest", "impliedVol", "bid", "ask",
    ]
    for column in numeric_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")

    required = ["strike", "expireYear", "expireMonth", "expireDay", "cp", "gamma", "openInterest"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise KeyError(f"Thiếu cột bắt buộc: {missing}")

    df["ticker"] = ticker.upper()
    df["option_type"] = df["cp"].astype(str).str.upper().map({"C": "call", "P": "put"})
    df["expiry"] = pd.to_datetime(
        {
            "year": df["expireYear"],
            "month": df["expireMonth"],
            "day": df["expireDay"],
        },
        errors="coerce",
    )
    df["openInterest"] = df["openInterest"].fillna(0).clip(lower=0)
    df = df.dropna(subset=["strike", "expiry", "option_type", "gamma"]).copy()
    df = df[df["openInterest"] >= min_open_interest].copy()

    if mm_position_mode == "always_short":
        df["mm_position"] = "short"
    elif mm_position_mode == "always_long":
        df["mm_position"] = "long"
    else:
        direction = df.apply(_trade_direction, axis=1)
        df["trade_direction"] = direction
        df["mm_position"] = np.where(direction.isin(["bid", "below-bid"]), "long", "short")

    unsigned_gex = (
        df["gamma"]
        * df["openInterest"]
        * CONTRACT_MULTIPLIER
        * spot_price
        * spot_price
        * MOVE_SIZE
    )
    short_sign = np.where(df["option_type"].eq("call"), 1.0, -1.0)
    position_sign = np.where(df["mm_position"].eq("short"), short_sign, -short_sign)

    df["spot"] = spot_price
    df["gex"] = unsigned_gex * position_sign
    df["abs_gex"] = df["gex"].abs()
    df["percent_from_spot"] = (df["strike"] - spot_price) / spot_price * 100
    df["mid"] = (df.get("bid", np.nan) + df.get("ask", np.nan)) / 2

    return df.sort_values(["strike", "expiry", "option_type"]).reset_index(drop=True)


options = normalize_and_calculate_gex(
    raw_options,
    ticker=TICKER,
    spot_price=spot,
    min_open_interest=MIN_OPEN_INTEREST,
    mm_position_mode=MM_POSITION_MODE,
)

print(f"Đã chuẩn hóa {len(options):,} contracts, {options['strike'].nunique():,} strikes, {options['expiry'].nunique():,} expirations.")
display(options.head())

Đã chuẩn hóa 10,222 contracts, 523 strikes, 31 expirations.


,strike,expireYear,expireMonth,expireDay,cp,gamma,delta,openInterest,impliedVol,bid,ask,ticker,option_type,expiry,trade_direction,mm_position,spot,gex,abs_gex,percent_from_spot,mid
0,204.78,2026,12,18,C,0.000004,0.999652,270,1.356912,537.04,540.37,QQQ,call,2026-12-18,mid,short,740.745,625.674214,625.674214,-72.354859,538.705
1,204.78,2026,12,18,P,0.000004,-0.000348,9462,0.802181,0.02,0.04,QQQ,put,2026-12-18,mid,short,740.745,-21926.405238,21926.405238,-72.354859,0.030
2,205.00,2027,1,15,C,0.000008,0.999246,240,1.165133,536.87,540.55,QQQ,call,2027-01-15,mid,short,740.745,1051.576676,1051.576676,-72.325159,538.710
3,205.00,2027,1,15,P,0.000008,-0.000754,3184,0.764056,0.06,0.09,QQQ,put,2027-01-15,mid,short,740.745,-13950.917239,13950.917239,-72.325159,0.075
4,205.00,2027,6,17,C,0.000021,0.997356,59,0.805515,540.54,544.50,QQQ,call,2027-06-17,mid,short,740.745,686.609847,686.609847,-72.325159,542.520


## 3. Tổng hợp toàn bộ Strike Profile

In [4]:
def build_strike_profile(options_df: pd.DataFrame, spot_price: float) -> pd.DataFrame:
    work = options_df.copy()
    is_call = work["option_type"].eq("call")
    is_put = work["option_type"].eq("put")

    work["call_gex_component"] = np.where(is_call, work["gex"], 0.0)
    work["put_gex_component"] = np.where(is_put, work["gex"], 0.0)
    work["call_oi_component"] = np.where(is_call, work["openInterest"], 0.0)
    work["put_oi_component"] = np.where(is_put, work["openInterest"], 0.0)
    work["call_contract_component"] = is_call.astype(int)
    work["put_contract_component"] = is_put.astype(int)

    profile = (
        work.groupby("strike", as_index=False)
        .agg(
            net_gex=("gex", "sum"),
            total_gex=("abs_gex", "sum"),
            call_gex=("call_gex_component", "sum"),
            put_gex=("put_gex_component", "sum"),
            call_oi=("call_oi_component", "sum"),
            put_oi=("put_oi_component", "sum"),
            call_contracts=("call_contract_component", "sum"),
            put_contracts=("put_contract_component", "sum"),
            expirations=("expiry", "nunique"),
        )
        .sort_values("strike")
        .reset_index(drop=True)
    )
    profile["total_oi"] = profile["call_oi"] + profile["put_oi"]
    profile["percent_from_spot"] = (profile["strike"] - spot_price) / spot_price * 100
    profile["ticker"] = TICKER.upper()
    profile["spot"] = spot_price
    profile["source_timestamp"] = source_timestamp

    ordered_columns = [
        "ticker", "source_timestamp", "spot", "strike", "percent_from_spot",
        "net_gex", "total_gex", "call_gex", "put_gex",
        "call_oi", "put_oi", "total_oi",
        "call_contracts", "put_contracts", "expirations",
    ]
    return profile[ordered_columns]


# Giới hạn toàn bộ dữ liệu dùng cho bảng/biểu đồ/export ở spot ± STRIKE_WINDOW.
raw_options = raw_options.loc[raw_options["strike"].between(spot - STRIKE_WINDOW, spot + STRIKE_WINDOW)].reset_index(drop=True)
options = options.loc[options["strike"].between(spot - STRIKE_WINDOW, spot + STRIKE_WINDOW)].reset_index(drop=True)
strike_profile = build_strike_profile(options, spot)
strike_profile = strike_profile.loc[
    strike_profile["strike"].between(spot - STRIKE_WINDOW, spot + STRIKE_WINDOW)
].reset_index(drop=True)

assert np.allclose(
    strike_profile["net_gex"],
    strike_profile["call_gex"] + strike_profile["put_gex"],
), "Net GEX không bằng Call GEX + Put GEX"

summary = pd.DataFrame(
    [{
        "ticker": TICKER.upper(),
        "spot": spot,
        "strikes": len(strike_profile),
        "contracts": len(options),
        "expirations": options["expiry"].nunique(),
        "net_gex": strike_profile["net_gex"].sum(),
        "total_gex": strike_profile["total_gex"].sum(),
        "call_gex": strike_profile["call_gex"].sum(),
        "put_gex": strike_profile["put_gex"].sum(),
        "call_oi": int(strike_profile["call_oi"].sum()),
        "put_oi": int(strike_profile["put_oi"].sum()),
    }]
)
display(summary.style.format({
    "spot": "${:,.2f}",
    "net_gex": "${:,.0f}",
    "total_gex": "${:,.0f}",
    "call_gex": "${:,.0f}",
    "put_gex": "${:,.0f}",
    "call_oi": "{:,.0f}",
    "put_oi": "{:,.0f}",
}))

,ticker,spot,strikes,contracts,expirations,net_gex,total_gex,call_gex,put_gex,call_oi,put_oi
0,QQQ,$740.75,200,4880,31,"$5,943,153,565","$18,948,490,379","$12,445,821,972","$-6,502,668,407","2,949,912","3,437,649"


## 4. Biểu đồ Strike Profile

In [10]:
if PLOT_MODE == "near":
    plot_profile = (
        strike_profile.assign(_distance=(strike_profile["strike"] - spot).abs())
        .nsmallest(PLOT_NEAREST_STRIKES, "_distance")
        .drop(columns="_distance")
        .sort_values("strike")
        .reset_index(drop=True)
    )
else:
    plot_profile = strike_profile.copy()

strike_diffs = np.diff(np.sort(plot_profile["strike"].unique()))
positive_diffs = strike_diffs[strike_diffs > 0]
bar_width = max(float(np.median(positive_diffs)) * 0.78, 0.05) if len(positive_diffs) else 0.8

print(
    f"Đang hiển thị {len(plot_profile):,}/{len(strike_profile):,} strikes "
    f"từ ${plot_profile['strike'].min():,.2f} đến ${plot_profile['strike'].max():,.2f}."
)

fig = go.Figure()
fig.add_bar(
    x=plot_profile["strike"],
    y=plot_profile["call_gex"] / 1_000_000,
    width=bar_width,
    name="Call Gamma",
    marker_color="#17be64",
    customdata=np.column_stack([
        plot_profile["net_gex"] / 1_000_000,
        plot_profile["total_gex"] / 1_000_000,
        plot_profile["call_oi"],
        plot_profile["put_oi"],
        plot_profile["percent_from_spot"],
    ]),
    hovertemplate=(
        "Strike $%{x}<br>Call GEX: $%{y:,.1f}M<br>"
        "Net GEX: $%{customdata[0]:,.1f}M<br>"
        "Total GEX: $%{customdata[1]:,.1f}M<br>"
        "Call OI: %{customdata[2]:,.0f}<br>"
        "Put OI: %{customdata[3]:,.0f}<br>"
        "From spot: %{customdata[4]:+.2f}%<extra></extra>"
    ),
)
fig.add_bar(
    x=plot_profile["strike"],
    y=plot_profile["put_gex"] / 1_000_000,
    width=bar_width,
    name="Put Gamma",
    marker_color="#f04662",
    hovertemplate="Strike $%{x}<br>Put GEX: $%{y:,.1f}M<extra></extra>",
)
fig.add_vline(x=spot, line_width=2, line_dash="dot", line_color="#3C8DF7")
fig.update_layout(
    title=f"{TICKER.upper()} Strike Profile — spot ${spot:,.2f}",
    xaxis_title="Strike",
    yaxis_title="Gamma Exposure ($M cho biến động 1%)",
    barmode="relative",
    template="plotly_dark",
    height=650,
    hovermode="x unified",
    bargap=0.08,
    xaxis=dict(
        range=[plot_profile["strike"].min() - bar_width, plot_profile["strike"].max() + bar_width],
        rangeslider=dict(visible=PLOT_MODE == "all"),
    ),
)
# Render bằng HTML để không phụ thuộc vào package nbformat.
display(HTML(fig.to_html(full_html=False, include_plotlyjs="cdn")))

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
chart_path = EXPORT_DIR / f"{TICKER.lower()}_strike_profile.html"
fig.write_html(chart_path, include_plotlyjs=True, full_html=True)
print(f"Đã lưu biểu đồ HTML: {chart_path.resolve()}")

Đang hiển thị 100/200 strikes từ $691.00 đến $790.00.


Đã lưu biểu đồ HTML: /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_strike_profile.html


## 5. Xuất toàn bộ dữ liệu

In [11]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
ticker_slug = TICKER.lower()
profile_path = EXPORT_DIR / f"{ticker_slug}_strike_profile.csv"
flow_profile_path = EXPORT_DIR / f"{ticker_slug}_strike_profile_with_flow.csv"
contracts_path = EXPORT_DIR / f"{ticker_slug}_option_contracts.csv"
raw_path = EXPORT_DIR / f"{ticker_slug}_raw_options.csv"
metadata_path = EXPORT_DIR / f"{ticker_slug}_metadata.json"

strike_profile.to_csv(profile_path, index=False)
if "strike_profile_with_flow" in globals():
    strike_profile_with_flow.to_csv(flow_profile_path, index=False)
options.to_csv(contracts_path, index=False)
raw_options.to_csv(raw_path, index=False)
metadata_payload = {
    "ticker": TICKER.upper(),
    "spot": spot,
    "source_timestamp": None if pd.isna(source_timestamp) else source_timestamp.isoformat(),
    "is_stale": page_data.get("isStale"),
    "source_url": source_url,
    "raw_option_count": int(len(raw_options)),
    "normalized_option_count": int(len(options)),
    "strike_count": int(len(strike_profile)),
    "expiration_count": int(options["expiry"].nunique()),
    "gex_formula": "gamma * open_interest * 100 * spot^2 * 0.01",
    "mm_position_mode": MM_POSITION_MODE,
    "volume_source": "pineify" if "pineify_df" in globals() and not pineify_df.empty else None,
    "volume_contracts_matched": int((options["volume"] > 0).sum()) if "volume" in options.columns else 0,
}
metadata_path.write_text(json.dumps(metadata_payload, indent=2, ensure_ascii=False), encoding="utf-8")

print("Đã xuất:")
exported_paths = [chart_path, profile_path, contracts_path, raw_path, metadata_path]
if "strike_profile_with_flow" in globals(): exported_paths.insert(2, flow_profile_path)

if "pineify_rendered_html" in globals() and "pineify_network" in globals():
    PINEIFY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    pineify_stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
    pineify_volume_path = PINEIFY_EXPORT_DIR / f"{ticker_slug}_volume_{pineify_stamp}.csv"
    pineify_html_path = PINEIFY_EXPORT_DIR / f"{ticker_slug}_rendered_{pineify_stamp}.html"
    pineify_network_path = PINEIFY_EXPORT_DIR / f"{ticker_slug}_network_{pineify_stamp}.json"
    pineify_df.to_csv(pineify_volume_path, index=False)
    pineify_html_path.write_text(pineify_rendered_html, encoding="utf-8")
    pineify_network_path.write_text(json.dumps(pineify_network, indent=2, ensure_ascii=False), encoding="utf-8")
    exported_paths.extend([pineify_volume_path, pineify_html_path, pineify_network_path])

for path in exported_paths:
    print(f"- {path.resolve()}")

Đã xuất:
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_strike_profile.html
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_strike_profile.csv
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_strike_profile_with_flow.csv
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_option_contracts.csv
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_raw_options.csv
- /Users/nguyenvokhang/Downloads/quant-research/exports/qqq_metadata.json
- /Users/nguyenvokhang/Downloads/quant-research/exports/pineify/qqq_volume_20260922T080549Z.csv
- /Users/nguyenvokhang/Downloads/quant-research/exports/pineify/qqq_rendered_20260922T080549Z.html
- /Users/nguyenvokhang/Downloads/quant-research/exports/pineify/qqq_network_20260922T080549Z.json
